# 資産運用向け Snowflake AI ハンズオン## Part 3: Cortex Search・Cortex Agent・Snowflake CoWorkCortex Search で文書を検索できるようにし、Cortex Analyst と組み合わせた **Cortex Agent** を作成します。**Agent Skills** で定型業務をパッケージ化し、Snowflake CoWork から使います。

### このパートで体験できること| 機能 | 用途 | 資産運用での活用例 ||---|---|---|| Cortex Search | 文書の意味検索 | 日本語の質問で英語の決算コールを検索 || 属性フィルタ | 検索結果の絞り込み | 経営陣の発言だけを検索 || Cortex Agent | ツールの自動選択 | 数値集計と文書検索を1つの質問で横断 || Agent Skills | 定型業務のパッケージ化 | 朝会メモ・決算レビューを毎回同じ品質で || Snowflake CoWork | 業務ユーザー向け対話UI | 自然言語で聞くだけで分析が返る |

### 体験ポイント> **「朝会用のブリーフィングメモを作って」の一言で、Agent が手順どおりに動く。**>> Agent はスキルの説明文を見て使うかを判断し、ステージ上の `SKILL.md` を読み込んで> Analyst と Search を順番に呼び出します。

### 前提条件- `part1_ai_functions.ipynb` と `part2_cortex_analyst.ipynb` が実行済み- `setup.sql` の Step 4 で `SKILL_STAGE` に Agent skills が搬入済み- ウェアハウスに `SNOW_AM_WH` を選択していること> ⏱️ **このパートの目安時間: 30分**

In [ ]:
-- 環境設定と前提条件の確認USE DATABASE SNOW_AM_DB;USE SCHEMA MARKET_INTELLIGENCE;USE WAREHOUSE SNOW_AM_WH;-- Part 1・Part 2 の成果物が揃っているか確認しますSELECT 'GOLD_EARNINGS_CALL_CHUNKS' AS "必要なオブジェクト", COUNT(*)::VARCHAR AS "件数"FROM GOLD_EARNINGS_CALL_CHUNKSUNION ALLSELECT 'GOLD_COMPANY_NEWS_ANALYZED', COUNT(*)::VARCHAR FROM GOLD_COMPANY_NEWS_ANALYZEDUNION ALLSELECT 'PORTFOLIO_MARKET_SEMANTIC_VIEW',       IFF(COUNT(*) > 0, '作成済み', '未作成')FROM SNOW_AM_DB.INFORMATION_SCHEMA.SEMANTIC_VIEWSWHERE NAME = 'PORTFOLIO_MARKET_SEMANTIC_VIEW';

In [ ]:
-- Agent skills がステージに搬入されているか確認-- SKILL.md は各スキルフォルダの「直下」にある必要がありますLS @SKILL_STAGE/ PATTERN = '.*SKILL\\.md';

## 1. Cortex Search Service を作成するCortex Search は非構造化テキストに対する検索サービスです。埋め込みモデルによるベクトル検索とキーワード検索を組み合わせたハイブリッド検索を、**インフラを何も用意せずに**使えます。### 設計のポイント| 項目 | 決算コール | ニュース ||---|---|---|| 検索対象列 | `CHUNK_TEXT`（発言本文） | `NEWS_CONTENT`（見出し + 本文） || 属性列（絞り込み用） | `TICKER`, `SPEAKER`, `SPEAKER_TYPE`, `FISCAL_PERIOD_LABEL` | `TICKER`, `SECTOR`, `SENTIMENT`, `EVENT_CATEGORY` || ID 列 | `CHUNK_ID`（自前で生成） | `NEWS_ID` || タイトル列 | `CHUNK_TITLE`（発言者 + 決算期） | `HEADLINE` || 埋め込みモデル | `snowflake-arctic-embed-l-v2.0`（多言語対応） | 同じ || 更新間隔 | `TARGET_LAG = '1 day'` | 同じ |> **⚠️ ノイズの除外が精度を左右する**>> 決算コール側の `AS (...)` 句に `SPEAKER_TYPE <> 'セクション'` という条件を入れています。> これは出席者一覧・免責事項などの**発言ではないチャンクを検索対象から外す**ためです。>> 実際にこの条件を入れる前は「AIインフラ投資が拡大している理由」という検索に対して> ページフッターが最上位にヒットしていました。検索対象を絞ることは、> 埋め込みモデルを変えるよりも効果が大きい場合があります。

In [ ]:
-- 決算コール用 Cortex Search Service の作成-- ⚠️ インデックス作成に1分程度かかりますCREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_EARNINGS_CALL    ON CHUNK_TEXT    ATTRIBUTES CHUNK_ID, TICKER, SPEAKER, SPEAKER_TYPE, FISCAL_PERIOD_LABEL    WAREHOUSE = SNOW_AM_WH    TARGET_LAG = '1 day'    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'    AS (        SELECT            TICKER || '-' || FISCAL_PERIOD_LABEL || '-' || LPAD(CHUNK_INDEX::VARCHAR, 3, '0') AS CHUNK_ID,            TICKER,            SPEAKER,            SPEAKER_TYPE,            FISCAL_PERIOD_LABEL,            CALL_DATE,            SPEAKER || '（' || FISCAL_PERIOD_LABEL || '）' AS CHUNK_TITLE,            CHUNK_TEXT        FROM GOLD_EARNINGS_CALL_CHUNKS        WHERE CHUNK_TEXT IS NOT NULL          AND LENGTH(TRIM(CHUNK_TEXT)) > 0          -- ページヘッダ・出席者一覧・免責事項などのノイズを除外します          AND SPEAKER_TYPE <> 'セクション'    );

In [ ]:
-- ニュース用 Cortex Search Service の作成CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_COMPANY_NEWS    ON NEWS_CONTENT    ATTRIBUTES NEWS_ID, TICKER, COMPANY_NAME_JA, SECTOR, SENTIMENT, EVENT_CATEGORY    WAREHOUSE = SNOW_AM_WH    TARGET_LAG = '1 day'    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'    AS (        SELECT            NEWS_ID,            TICKER,            COMPANY_NAME_JA,            SECTOR,            SENTIMENT,            EVENT_CATEGORY,            HEADLINE,            SOURCE,            URL,            TO_TIMESTAMP_NTZ(PUBLISHED_AT) AS PUBLISHED_AT,            HEADLINE || '\n\n' || BODY AS NEWS_CONTENT        FROM GOLD_COMPANY_NEWS_ANALYZED        WHERE BODY IS NOT NULL AND LENGTH(TRIM(BODY)) > 0    );

In [ ]:
-- 作成したサービスの状態を確認SHOW CORTEX SEARCH SERVICES IN SCHEMA SNOW_AM_DB.MARKET_INTELLIGENCE;

In [ ]:
-- インデックス状態を整形して表示（ACTIVE になっていることを確認）SELECT "name" AS "サービス名",       "search_column" AS "検索対象列",       "indexing_state" AS "インデックス状態",       "serving_state" AS "提供状態",       "target_lag" AS "更新間隔"FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

> **💡 参考: AI Studio の GUI から作成する場合**>> 時間に余裕があれば、GUI からの作成も体験してみてください。>> 1. Snowsight のナビゲーションメニューから **AI と ML** » **検索** を開く> 2. **作成** をクリック> 3. 下表を設定して **作成** をクリック

| 設定項目 | 値 ||---|---|| データベース | `SNOW_AM_DB` || スキーマ | `MARKET_INTELLIGENCE` || サービス名 | `SEARCH_EARNINGS_CALL_GUI` || インデックス対象テーブル | `GOLD_EARNINGS_CALL_CHUNKS` || 検索列 | `CHUNK_TEXT` || 属性列 | `TICKER`, `SPEAKER`, `SPEAKER_TYPE` || サービスに含む列 | すべて選択 || 更新間隔 | `1 day` || 埋め込みモデル | `snowflake-arctic-embed-l-v2.0` || ウェアハウス | `SNOW_AM_WH` |> **💡 GUI と SQL の違い**>> GUI では `AS (...)` の絞り込み条件を書けないため、ノイズ除外はテーブル側で> 対応する必要があります。定義をコード管理したい場合も SQL のほうが適しています。

### 2-1. キーワード検索 vs セマンティック検索決算コールの原文は**英語**です。一方、運用担当者が入力するのは**日本語**です。まずキーワード検索（`LIKE`）を試してみましょう。

In [ ]:
-- A: キーワード検索（LIKE）で日本語クエリSELECT COUNT(*) AS "ヒット件数"FROM GOLD_EARNINGS_CALL_CHUNKSWHERE CHUNK_TEXT LIKE '%データセンター需要%'   OR CHUNK_TEXT LIKE '%AIインフラ投資%';-- 結果: 0件（原文が英語なので、日本語のキーワードでは1件も当たりません）

In [ ]:
-- B: セマンティック検索（同じ意図を日本語で問う）SELECT ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度",       f.value:SPEAKER::STRING AS "発言者",       f.value:SPEAKER_TYPE::STRING AS "区分",       LEFT(REGEXP_REPLACE(f.value:CHUNK_TEXT::STRING, '\\s+', ' '), 400) AS "本文（先頭400字）"FROM (    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(        'SNOW_AM_DB.MARKET_INTELLIGENCE.SEARCH_EARNINGS_CALL',        '{            "query": "AIインフラ投資が拡大している理由を経営陣はどう説明しているか",            "columns": ["SPEAKER", "SPEAKER_TYPE", "CHUNK_TEXT"],            "limit": 3        }'    ) AS result_json),LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;-- 結果: データセンター事業の需要について語っている箇所がヒットします

> **💡 比較のポイント**>> - キーワード検索は **0件**。日本語で英語文書を探すことは原理的にできません> - セマンティック検索は**言語をまたいで**該当箇所を見つけます。>   `snowflake-arctic-embed-l-v2.0` が多言語対応の埋め込みモデルだからです>> **実務上の注意**>> 言語をまたぐ検索は機能しますが、**同一言語同士の検索よりスコアは低めに出ます**。> 類似度が 0.4〜0.6 程度でも十分に的を射た結果が返ることがあるため、> スコアの絶対値で閾値を切るのは避けたほうが安全です。>> より高い精度が必要な場合は、以下の選択肢があります。>> - 検索前にクエリを英訳する（`SNOWFLAKE.CORTEX.TRANSLATE`）> - 文書側を和訳した列を用意して両方をインデックスする> - `TEXT INDEXES` と `VECTOR INDEXES` を併用したマルチインデックス検索を使う### 2-2. 属性フィルタで絞り込むPart 1 で付与した `SPEAKER_TYPE` を使い、**経営陣の発言だけ**を検索します。「アナリストの質問ではなく、会社側の見解を知りたい」という実務要件に対応できます。

In [ ]:
-- 属性フィルタ: 経営陣の発言のみを検索SELECT ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度",       f.value:SPEAKER::STRING AS "発言者",       f.value:SPEAKER_TYPE::STRING AS "区分",       LEFT(REGEXP_REPLACE(f.value:CHUNK_TEXT::STRING, '\\s+', ' '), 400) AS "本文（先頭400字）"FROM (    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(        'SNOW_AM_DB.MARKET_INTELLIGENCE.SEARCH_EARNINGS_CALL',        '{            "query": "次四半期の売上ガイダンスと売上総利益率の見通し",            "columns": ["SPEAKER", "SPEAKER_TYPE", "CHUNK_TEXT"],            "filter": {"@eq": {"SPEAKER_TYPE": "経営陣"}},            "limit": 3        }'    ) AS result_json),LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 2-3. ニュースを検索するニュース側では、Part 1 で AI が付与した `SENTIMENT` と `EVENT_CATEGORY` が属性として使えます。「非構造化データから AI が作った属性で、非構造化データを絞り込む」という構図です。

In [ ]:
-- ニュース検索: 供給網の問題に関する報道SELECT ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度",       f.value:COMPANY_NAME_JA::STRING AS "企業",       f.value:SECTOR::STRING AS "セクター",       f.value:SENTIMENT::STRING AS "センチメント",       f.value:EVENT_CATEGORY::STRING AS "イベント種別",       f.value:HEADLINE::STRING AS "見出し"FROM (    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(        'SNOW_AM_DB.MARKET_INTELLIGENCE.SEARCH_COMPANY_NEWS',        '{            "query": "供給網の問題や出荷遅延、供給制約",            "columns": ["COMPANY_NAME_JA", "SECTOR", "SENTIMENT", "EVENT_CATEGORY", "HEADLINE"],            "limit": 5        }'    ) AS result_json),LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ニュース検索 + センチメントフィルタ: ネガティブな報道のみSELECT ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度",       f.value:COMPANY_NAME_JA::STRING AS "企業",       f.value:SENTIMENT::STRING AS "センチメント",       f.value:HEADLINE::STRING AS "見出し"FROM (    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(        'SNOW_AM_DB.MARKET_INTELLIGENCE.SEARCH_COMPANY_NEWS',        '{            "query": "規制対応やコスト増加の懸念",            "columns": ["COMPANY_NAME_JA", "SENTIMENT", "HEADLINE"],            "filter": {"@eq": {"SENTIMENT": "ネガティブ"}},            "limit": 5        }'    ) AS result_json),LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## 3. Agent Skills を理解するここまでで Agent に持たせる**道具（tools）**が揃いました。| 道具 | できること ||---|---|| Cortex Analyst | 数値を集計する || Cortex Search | 文書を検索する |しかし道具があるだけでは、**「朝会用のブリーフィングメモを作って」**という依頼に毎回同じ品質で応えることはできません。どのツールをどの順番で呼び、どんなフォーマットで出力するかが決まっていないからです。**Agent Skills** はこれを解決します。定型業務の手順・出力フォーマット・禁止事項を`SKILL.md` に書いておくと、Agent が該当する依頼を受けたときに自動で読み込んで従います。

### SKILL.md の構造```---name: スキル名（一意）description: どんなときに使うスキルかの説明。★Agent はこれを見て使うか判断する---（本文 = Agent への詳細な指示）## 実行手順## 出力フォーマット## 禁止事項```> **⚠️ 最重要ポイント**>> Agent が「このスキルを使うべきか」を判断する材料は **`description` だけ**です。> 本文（instructions）は、使うと決めたあとに初めて読み込まれます。>> したがって `description` には、**ユーザーが実際に使いそうな言い回しを列挙**しておく> 必要があります。本ハンズオンの `morning-brief` では次のように書いています。>> > 「朝会資料」「モーニングブリーフ」「今日の注目銘柄」「朝のまとめ」> > 「運用会議用のメモ」「保有銘柄の状況をまとめて」などの依頼時に使用する。

In [ ]:
-- ステージ上の SKILL.md の中身を確認する-- Markdown をそのまま1つの文字列として読み込むためのファイルフォーマットを作りますCREATE OR REPLACE FILE FORMAT MD_RAW_FORMAT    TYPE = CSV    FIELD_DELIMITER = NONE    RECORD_DELIMITER = NONE    SKIP_HEADER = 0;-- Agent が実際に読み込むのは、このステージ上のファイルですSELECT LEFT($1, 1200) AS "morning-brief / SKILL.md（先頭1200字）"FROM @SKILL_STAGE/skills/morning-brief/SKILL.md (FILE_FORMAT => MD_RAW_FORMAT);

> **💡 skills の格納場所について**>> Agent skills は **名前付きステージ**または **Git リポジトリ**から参照できます。> 本ハンズオンでは `setup.sql` の Step 4 で、Git リポジトリから `COPY FILES` で> ステージにコピーする方式を採っています。>> | 方式 | 指定例 | 特徴 |> |---|---|---|> | ステージ | `@DB.SCHEMA.SKILL_STAGE/skills/morning-brief` | ロールで権限管理できる |> | Git（タグ参照） | `@DB.SCHEMA.REPO/tags/latest/skills/morning-brief` | `FETCH` で自動更新される |> | Git（コミット参照） | `@DB.SCHEMA.REPO/commits/abc123/skills/morning-brief` | 不変。本番向け |>> **制約**: `SKILL.md` はスキルフォルダの**直下**に置く必要があります。> サブディレクトリは探索されません。補助スクリプトも同じフォルダに置きます。>> スキルの内容を更新したい場合は、ステージ上のファイルを差し替えるだけでよく、> Agent の再作成は不要です。Agent はファイルをコピーせず、**参照している**だけだからです。

## 4. Cortex Agent を作成するいよいよ Agent を作ります。構成は以下のとおりです。```MARKET_INTELLIGENCE_AGENT├── instructions│   ├── orchestration …… どのツールをどう選ぶか│   └── response      …… どう答えるか（口調・数値書式・禁止事項）├── tools│   ├── PORTFOLIO_MARKET_ANALYST  (cortex_analyst_text_to_sql)│   ├── EARNINGS_CALL_SEARCH      (cortex_search)│   └── COMPANY_NEWS_SEARCH       (cortex_search)└── skills    ├── morning-brief    (STAGE)    └── earnings-review  (STAGE)```

### 押さえておきたい制約> **`ALTER AGENT` でツールを追加することはできません。**>> ツールやスキルを変更する方法は次の4つです。>> | 方法 | 内容 |> |---|---|> | `CREATE OR REPLACE AGENT` | 定義全体を書き直す（本ハンズオンの方式） |> | `ALTER AGENT ... MODIFY LIVE VERSION SET SPECIFICATION` | 仕様全体を差し替える。**変更しない項目も全部書く必要あり** |> | AI Studio（GUI） | 画面上でツール・スキルを追加・編集する |> | REST API | プログラムから更新する |>> 定義をコード管理するなら `CREATE OR REPLACE` が最も扱いやすいです。

### instructions の書き方`orchestration` と `response` で役割を分けます。| 項目 | 書く内容 ||---|---|| `orchestration` | ツール選択の基準、複合質問への対応、データ期間の注意、整合性チェック || `response` | 口調、数値の書式、出典の明示、可視化、禁止事項 |特に**禁止事項**は重要です。金融の文脈では「投資助言と受け取られる表現をしない」「データにない数値を推測で埋めない」といった制約を明示的に書く必要があります。

In [ ]:
-- Cortex Agent の作成CREATE OR REPLACE AGENT SNOW_AM_DB.MARKET_INTELLIGENCE.MARKET_INTELLIGENCE_AGENTWITH PROFILE = '{"display_name": "マーケットインテリジェンスAgent", "color": "blue"}'COMMENT = 'スノーアセットマネジメント（架空）の運用部門向けAIアシスタント。ポートフォリオ・市場データ・決算コール・ニュースを横断して分析する。'FROM SPECIFICATION $$models:  orchestration: autoinstructions:  orchestration: |    あなたはスノーアセットマネジメントの運用部門を支援する分析アシスタントです。    ユーザーの質問に対して、以下の基準でツールを選択してください。    【ツール選択の基準】    - 保有明細・株価・財務諸表・ニュース件数などの「数値の集計」      → PORTFOLIO_MARKET_ANALYST（Cortex Analyst）を使用    - 決算カンファレンスコールで「経営陣が何を語ったか」      → EARNINGS_CALL_SEARCH（Cortex Search）を使用    - 個別ニュースの「内容や文脈」      → COMPANY_NEWS_SEARCH（Cortex Search）を使用    【複合的な質問への対応】    「決算コールの内容と実績数値を突き合わせて」のように定性情報と定量情報の    両方が必要な質問では、Search と Analyst を両方呼び出して統合してください。    片方だけで答えないこと。    【データ期間についての注意】    - 株価データは直近3年分です    - 財務諸表は SEC 提出書類ベースで、企業によって最新四半期が異なります    - JPMorgan Chase（JPM）は売上（Revenues）の XBRL タグを使用していないため、      売上は NULL になります。純利益と EPS は取得できます    - 決算カンファレンスコールは NVIDIA の Q1 FY2027（2026年5月20日発表）のみです    【整合性チェック】    回答を返す前に以下を確認してください。    - 決算コールの記述は概数（例: 約820億ドル）であり、SEC 実績の厳密値      （例: 81,615百万ドル）とは表現が異なります。「一致」と断定せず、      「概数として整合している」と表現してください    - 検索結果が得られなかった場合は、その旨を明示してください  response: |    以下のルールに従って回答してください。    【口調・スタイル】    - 丁寧語（です・ます調）で回答する    - 結論を先に述べ、そのあとに根拠を示す    【数値の表示】    - 金額は3桁区切りで表示する（例: 1,234,567 ドル）    - 大きな金額は百万ドル単位に丸めて併記する（例: 81,615 百万ドル）    - パーセンテージは小数点第1位まで表示する（例: 12.3%）    - 騰落率は符号を明示する（例: +3.2%、-1.8%）    - 日付は YYYY年MM月DD日 形式で表示する    【出典の明示】    - Cortex Search で取得した決算コールやニュースの内容を使う場合は、      発言者名またはニュース見出しを必ず併記する    - 数値を示す場合は、それが SEC 実績か決算コールの記述かを区別する    【可視化】    - 時系列の推移や構成比を示す場合は、積極的にグラフを生成する    【禁止事項】    - 投資助言と受け取られる断定的な表現をしない。      「買うべき」「売るべき」「推奨する」は使わず、「着目点」として論点を提示する    - データにない数値を推測で埋めない。取得できなかった項目は「データ未取得」と明示する    - 出典のない発言の要約をしない    - 特定の顧客名・実在の金融機関名を回答に含めないtools:  - tool_spec:      type: cortex_analyst_text_to_sql      name: PORTFOLIO_MARKET_ANALYST      description: |        自社ファンドの保有明細、米国株の日次株価、SEC提出書類ベースの四半期財務諸表、        AI分析済み企業ニュースを対象に、自然言語から SQL を生成して数値を集計します。        取得可能: 保有時価・構成比・AUM・始値/高値/安値/終値・出来高・売上・粗利・営業利益・        純利益・希薄化後EPS・研究開発費・営業利益率・粗利率・ニュース件数・センチメント別集計        使用例: 「ファンド別のAUMを教えて」「NVDAの四半期売上の推移」「営業利益率上位3銘柄」  - tool_spec:      type: cortex_search      name: EARNINGS_CALL_SEARCH      description: |        決算カンファレンスコールの文字起こしを発言者単位で検索します。        対象: NVIDIA Q1 FY2027（2026年5月20日発表）の決算コール全文        発言者区分（経営陣 / アナリスト）で絞り込めます。        使用例: 「経営陣が語ったデータセンター需要の見通し」「次四半期のガイダンス」        「アナリストからの質問内容」  - tool_spec:      type: cortex_search      name: COMPANY_NEWS_SEARCH      description: |        米国大型株10銘柄に関する企業ニュースを検索します。        対象: 2026年1月〜6月のニュース50件。AIが付与したセンチメントとイベント種別を保持        使用例: 「サプライチェーンに関するネガティブなニュース」「規制関連の報道」        「NVIDIAの決算前後のニュース」tool_resources:  PORTFOLIO_MARKET_ANALYST:    semantic_view: SNOW_AM_DB.MARKET_INTELLIGENCE.PORTFOLIO_MARKET_SEMANTIC_VIEW    execution_environment:      type: warehouse      warehouse: SNOW_AM_WH      query_timeout: 120  EARNINGS_CALL_SEARCH:    name: SNOW_AM_DB.MARKET_INTELLIGENCE.SEARCH_EARNINGS_CALL    max_results: 5    id_column: CHUNK_ID    title_column: CHUNK_TITLE  COMPANY_NEWS_SEARCH:    name: SNOW_AM_DB.MARKET_INTELLIGENCE.SEARCH_COMPANY_NEWS    max_results: 5    id_column: NEWS_ID    title_column: HEADLINEskills:  - name: morning-brief    source:      type: STAGE      path: '@SNOW_AM_DB.MARKET_INTELLIGENCE.SKILL_STAGE/skills/morning-brief'  - name: earnings-review    source:      type: STAGE      path: '@SNOW_AM_DB.MARKET_INTELLIGENCE.SKILL_STAGE/skills/earnings-review'$$;

In [ ]:
-- 作成した Agent の定義を確認DESCRIBE AGENT SNOW_AM_DB.MARKET_INTELLIGENCE.MARKET_INTELLIGENCE_AGENT;

In [ ]:
-- ツールとスキルが登録されているかを整形して確認WITH spec AS (    SELECT PARSE_JSON("agent_spec") AS s    FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())))SELECT 'ツール' AS "種別",       t.value:tool_spec:name::VARCHAR AS "名称",       t.value:tool_spec:type::VARCHAR AS "タイプ"FROM spec, LATERAL FLATTEN(input => spec.s:tools) tUNION ALLSELECT 'スキル',       k.value:name::VARCHAR,       k.value:source:type::VARCHAR || ' : ' || k.value:source:path::VARCHARFROM spec, LATERAL FLATTEN(input => spec.s:skills) kORDER BY "種別" DESC, "名称";

## 5. Snowflake CoWork で実践するAgent に設定した tools と skills は、**Snowflake CoWork から自動的に利用可能**になります。追加の設定は不要です。### アクセス手順1. Snowsight の左下のメニューから **Snowflake CoWork** を選ぶ   （または `https://app.snowflake.com/<org>/<account>/#/cowork` を開く）2. 画面上部の Agent 選択から **マーケットインテリジェンスAgent** を選ぶ3. ロールとウェアハウスが `ACCOUNTADMIN` / `SNOW_AM_WH` になっていることを確認する

### シナリオ1: 単一ツールで答えられる質問```当社ファンドのAUM上位5銘柄とセクター構成を教えて```数値の集計だけで答えられるため、**Cortex Analyst だけ**が呼ばれます。グラフが自動生成されることも確認してください。続けて掘り下げてみましょう。```セクター別の構成比を円グラフで見せて``````ファンドごとの銘柄数と平均構成比の違いを教えて```

### シナリオ2: 複数ツールを横断する質問```NVDAの直近決算コールで経営陣が語ったデータセンター需要の見通しと、実績売上の推移を突き合わせて```この質問には定性情報（決算コール）と定量情報（財務諸表）の両方が必要です。**Cortex Search と Cortex Analyst が両方**呼ばれることを確認してください。さらにこんな質問も試せます。```ネガティブなニュースが出ている保有銘柄について、何が懸念されているのか具体的に教えて``````NVIDIAの次四半期ガイダンスはいくらで、それは直近実績から見て何%の成長になる？```

### シナリオ3: Agent Skill が発火する質問```朝会用のブリーフィングメモを作って```これが本パートの見せ場です。Agent は次のように動きます。1. 依頼が `morning-brief` スキルの `description` に合致すると判断する2. ステージから `SKILL.md` を読み込む3. スキルに書かれた手順どおりに、Analyst と Search を順番に呼び出す4. スキルで指定した固定フォーマットで出力するもう1つのスキルも試してみましょう。```NVIDIAの決算レビューメモを作って```こちらは `earnings-review` スキルが発火し、決算コールの記述と SEC 実績を突き合わせたメモが生成されます。

### 期待されるツール／スキルの動作質問に対して何が呼ばれるかを、あらかじめ整理しておきます。実際の動作と照らし合わせてみてください。| 質問 | 呼ばれるもの ||---|---|| 当社ファンドのAUM上位5銘柄とセクター構成を教えて | `PORTFOLIO_MARKET_ANALYST` || 経営陣はデータセンター需要をどう見ている？ | `EARNINGS_CALL_SEARCH` || サプライチェーンに関するネガティブなニュースは？ | `COMPANY_NEWS_SEARCH` || 決算コールの内容と実績売上を突き合わせて | `EARNINGS_CALL_SEARCH` + `PORTFOLIO_MARKET_ANALYST` || 朝会用のブリーフィングメモを作って | **`morning-brief` スキル** + Analyst + Search || NVIDIAの決算レビューメモを作って | **`earnings-review` スキル** + Search + Analyst |

> **💡 スキルが発火したことを確認する方法**>> CoWork では回答生成中に **thinking steps（思考の過程）**が表示されます。> ここにスキルの読み込みが現れます。>> ```> The user wants a morning briefing memo.> This matches the morning-brief skill. Let me load it.> ```>> このあと `Reading skills` というステータスが表示され、続いて Analyst / Search の> 呼び出しが並びます。ここまで確認できれば、スキルは正しく動作しています。>> **スキルが発火しない場合の対処**>> 1. `DESCRIBE AGENT` の出力に `skills` が2件含まれているか確認する> 2. `LS @SKILL_STAGE/ PATTERN='.*SKILL\.md'` で `skills/<名前>/SKILL.md` の>    階層になっているか確認する> 3. チャット入力欄の **+** ボタンからスキルを**明示的に選択**する> 4. それでも発火しないなら、`SKILL.md` の `description` にユーザーの言い回しを追加する

## 6. モニタリングAgent の利用状況とコストを把握します。> **⚠️ 重要: 利用履歴は API 経由と UI 経由で別のビューに記録される**>> | ビュー | 記録される利用 |> |---|---|> | `CORTEX_AGENT_USAGE_HISTORY` | REST API 経由での Agent 呼び出し |> | `SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY` | Snowflake CoWork（UI）経由での呼び出し |>> 片方だけを見ていると利用実態を見誤ります。両方を確認してください。> なお `ACCOUNT_USAGE` のビューには**最大数時間の反映遅延**があります。

In [ ]:
-- Q1: API 経由の Agent 呼び出し履歴SELECT CONVERT_TIMEZONE('Asia/Tokyo', START_TIME) AS "実行時刻（JST）",       USER_NAME AS "ユーザー",       AGENT_NAME AS "エージェント",       TOKENS AS "トークン数",       ROUND(TOKEN_CREDITS, 6) AS "クレジット",       METADATA:role_name::VARCHAR AS "ロール"FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORYWHERE AGENT_NAME = 'MARKET_INTELLIGENCE_AGENT'ORDER BY START_TIME DESCLIMIT 20;

In [ ]:
-- Q2: Snowflake CoWork（UI）経由の呼び出し履歴SELECT CONVERT_TIMEZONE('Asia/Tokyo', START_TIME) AS "実行時刻（JST）",       USER_NAME AS "ユーザー",       AGENT_NAME AS "エージェント",       TOKENS AS "トークン数",       ROUND(TOKEN_CREDITS, 6) AS "クレジット"FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWFLAKE_INTELLIGENCE_USAGE_HISTORYWHERE AGENT_NAME = 'MARKET_INTELLIGENCE_AGENT'ORDER BY START_TIME DESCLIMIT 20;

In [ ]:
-- Q3: API と UI を合算した日次クレジット消費WITH combined AS (    SELECT START_TIME, USER_NAME, AGENT_NAME, TOKENS, TOKEN_CREDITS, 'API' AS CHANNEL    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY    WHERE AGENT_NAME = 'MARKET_INTELLIGENCE_AGENT'    UNION ALL    SELECT START_TIME, USER_NAME, AGENT_NAME, TOKENS, TOKEN_CREDITS, 'CoWork(UI)'    FROM SNOWFLAKE.ACCOUNT_USAGE.SNOWFLAKE_INTELLIGENCE_USAGE_HISTORY    WHERE AGENT_NAME = 'MARKET_INTELLIGENCE_AGENT')SELECT DATE(CONVERT_TIMEZONE('Asia/Tokyo', START_TIME)) AS "日付（JST）",       CHANNEL AS "経路",       COUNT(*) AS "呼び出し回数",       SUM(TOKENS) AS "トークン数",       ROUND(SUM(TOKEN_CREDITS), 6) AS "クレジット"FROM combinedGROUP BY ALLORDER BY "日付（JST）" DESC, "経路";

### 観察された問題と改善アクションモニタリングと実際の対話から得られる気づきを、次の改善につなげます。| 観察された問題 | 改善アクション ||---|---|| 間違ったツールが選択される | `orchestration` の指示と、各ツールの `description` を見直す || SQL が不正確・意図と違う | セマンティックビューの `METRICS` と `WITH SYNONYMS` を追加・修正する || 検索結果が的外れ | 検索対象からノイズを除外する、`max_results` を増やす、チャンクサイズを見直す || スキルが発火しない | `SKILL.md` の `description` にユーザーの言い回しを追加する || 出力フォーマットが毎回違う | スキルの「出力フォーマット」節をより具体的に書く || 回答が冗長 | `response` の指示に文字数や構成の制約を追加する || 投資助言的な表現が出る | `response` の禁止事項を強化する || クレジット消費が多い | `max_results` を絞る、extended thinking の常用を避ける |> **💡 運用のコツ**>> Agent の品質改善は、Agent の定義だけをいじっても限界があります。> 実際には **セマンティックビューの定義** と **検索対象データの前処理** が> 効いてくる場面が多いです。Part 1 のノイズ除去や Part 2 の同義語定義が、> ここで結果として返ってきます。

## まとめ### 作成したオブジェクト| オブジェクト | 内容 ||---|---|| `SEARCH_EARNINGS_CALL` | 決算コールを発言者単位で検索（ノイズ除外済み） || `SEARCH_COMPANY_NEWS` | 企業ニュースを検索（センチメント・イベント種別で絞り込み可能） || `MARKET_INTELLIGENCE_AGENT` | Analyst 1本 + Search 2本 + Skills 2本 |

### このパートで押さえたポイント1. **検索対象からノイズを除外することが精度に効く**。埋め込みモデルを変えるより効果的な場合がある2. **セマンティック検索は言語をまたぐ**。日本語クエリで英語文書を検索できる。ただしスコアは低めに出る3. **AI が付与した属性が、そのまま検索の絞り込み条件になる**4. **Agent がスキルを使うか判断する材料は `description` だけ**。言い回しを列挙しておく5. **`ALTER AGENT` でツール追加はできない**。`CREATE OR REPLACE` で定義全体を管理する6. **利用履歴は API 経由と UI 経由で別のビュー**。両方を見る必要がある

### ハンズオン全体で作ったもの```Marketplace の公開データ（縦持ち）        │ setup.sql: ピボットして分析可能な形に        ▼構造化データ（銘柄・株価・財務・為替・金利・保有明細）        +非構造化データ（決算コール PDF・ニュース CSV）        │ Part 1: AI_PARSE_DOCUMENT / AI_SENTIMENT / AI_CLASSIFY / AI_EXTRACT / AI_AGG        ▼Gold 層（チャンク・数値ファクト・AI分析済みニュース）        │ Part 2: セマンティックビューで業務用語と指標を定義        ▼Cortex Analyst（自然言語 → SQL）        +Cortex Search（自然言語 → 文書）        │ Part 3: Agent がツールを選び、Skill が手順を規定        ▼Snowflake CoWork（業務ユーザーが自然言語で使える状態）```

### 次に何をするかこのハンズオンで作った構成は、そのまま業務展開の出発点になります。| やりたいこと | 次のステップ ||---|---|| 対象銘柄を増やす | `setup.sql` の Step 6 のティッカーリストを変更する || 自社の保有データを使う | `FACT_FUND_HOLDING` を実データに差し替える || 決算コールを増やす | `data/earnings_calls/` に PDF を追加して Part 1 を再実行する || 定型業務を追加する | `skills/` に新しい `SKILL.md` を追加する || 権限を分ける | ロールごとに Semantic View / Search / Agent の USAGE を制御する || データを自動更新する | Marketplace データは自動更新される。Gold 層はタスクで定期実行する |### 後片付けハンズオン終了後、`cleanup.sql` を実行すると作成したデータベースとウェアハウスを削除できます。